<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Receta de la Cosecha: Varios Ingredientes a la Vez 🍲
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/04%20-%20No%20Parametrica%20Regresion%20y%20Etica/Para%20Dummies/02_Regresion_Multiple_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no matemáticos»** del [cuaderno 02 del Módulo 04 (Regresión múltiple y diagnóstico)](../02_Regresion_Multiple_y_Diagnostico.ipynb). La cosecha no depende de una sola cosa: depende del **abono**, de la **lluvia**, de si hay **riego**, de la **altura** de la finca... Es como una **receta**: queremos saber **cuánto aporta cada ingrediente**, *manteniendo los otros quietos*.

| Lo que decimos | Lo que significa en la vida diaria |
|---|---|
| **Ingrediente (variable)** | Cada cosa que puede influir en la cosecha: abono, lluvia, riego... |
| **Aporte de un ingrediente** | Cuánto cambia la cosecha, en promedio, si ese ingrediente sube una unidad **y todo lo demás se queda igual** |
| **Cocineros repetidos** | Dos ingredientes que casi dicen lo mismo (la altura y la temperatura): el modelo no sabe a cuál dar el crédito |
| **Dato raro** | Una parcela con un número mal anotado que puede mover toda la línea |

Al terminar podrás explicar:
1. Por qué el **aporte del abono** cambia cuando también miras el riego.
2. Qué pasa cuando **dos ingredientes son casi el mismo** (los aportes se vuelven inestables).
3. Cómo **un solo dato mal anotado** mueve el resultado y qué hacer.

> 🔗 Si luego quieres ver la versión **con matrices, VIF, distancia de Cook y errores robustos**: [cuaderno estándar](../02_Regresion_Multiple_y_Diagnostico.ipynb).

---
## 1. El Abono se Lleva el Crédito del Riego 💧

Tenemos **150 fincas** inventadas con su **abono**, **lluvia**, si tienen **riego** (1 = sí) y su **cosecha**. En las fincas que abonan más también es más común que haya riego. Si miras solo el abono, el abono se lleva **también el crédito del riego**. La receta completa (varios ingredientes a la vez) los separa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)                      # semilla fija: siempre salen los mismos numeros
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

g = np.random.default_rng(11)
n = 150
abono = g.uniform(20, 140, n)                                              # kg/ha
lluvia = g.normal(120, 30, n)                                              # mm al mes
riego = (g.random(n) < 1 / (1 + np.exp(-(abono - 80) / 25))).astype(float)  # mas abono -> mas probable tener riego
altitud = g.uniform(2500, 3200, n)                                         # metros sobre el nivel del mar
temperatura = 28 - 0.0065 * altitud + g.normal(0, 0.15, n)                 # casi determinada por la altitud
cosecha = 10 + 0.05 * abono + 0.03 * lluvia + 3.0 * riego - 0.004 * (altitud - 2800) + g.normal(0, 2, n)
fincas = pd.DataFrame(dict(cosecha=cosecha, abono=abono, lluvia=lluvia, riego=riego, altitud=altitud, temperatura=temperatura))

def ajusta(columnas, datos=fincas):
    """Aporte de cada ingrediente (minimos cuadrados)."""
    X = np.c_[np.ones(len(datos)), datos[columnas].to_numpy()]
    return dict(zip(["base"] + columnas, np.linalg.lstsq(X, datos["cosecha"].to_numpy(), rcond=None)[0]))

solo_abono = ajusta(["abono"])
receta = ajusta(["abono", "lluvia", "riego"])
print(f"Mirando SOLO el abono: cada 10 kg de abono -> +{10 * solo_abono['abono']:.2f} t/ha")
print(f"Receta completa (abono + lluvia + riego): cada 10 kg de abono -> +{10 * receta['abono']:.2f} t/ha | tener riego -> +{receta['riego']:.2f} t/ha | cada 10 mm de lluvia -> +{10 * receta['lluvia']:.2f} t/ha")
print(f"(Los valores verdaderos con que se inventaron los datos: abono +0.50 por 10 kg, riego +3.00, lluvia +0.30 por 10 mm)")
print(f"Relacion entre abono y riego: las fincas con riego abonan en promedio {fincas.loc[fincas.riego == 1, 'abono'].mean():.0f} kg y las sin riego {fincas.loc[fincas.riego == 0, 'abono'].mean():.0f} kg")

plt.figure(figsize=(9, 4))
for r, c, nombre in [(0, "#94a3b8", "sin riego"), (1, "#2563eb", "con riego")]:
    sub = fincas[fincas.riego == r]; plt.scatter(sub.abono, sub.cosecha, color=c, label=nombre, s=22)
xs = np.array([20, 140]); plt.plot(xs, solo_abono["base"] + solo_abono["abono"] * xs, color="black", ls="--", label="solo abono")
plt.xlabel("abono (kg/ha)"); plt.ylabel("cosecha (t/ha)"); plt.title("Las fincas con mas abono tambien tienen mas riego"); plt.legend(); plt.tight_layout(); plt.show()

assert solo_abono["abono"] > receta["abono"] * 1.15                         # el abono 'solo' exagera su aporte
assert abs(receta["riego"] - 3.0) < 1.2 and abs(receta["abono"] - 0.05) < 0.02

---
**Lección:** mirando **solo el abono**, cada 10 kg parecen traer **+0.81 t/ha**; con la **receta completa** el aporte baja a **+0.61 t/ha** (el valor con que se inventaron los datos era +0.50; las estimaciones siempre tienen algo de error). La diferencia ocurre porque las fincas **con riego abonan más** (97 kg en promedio contra 56 kg de las que no tienen riego) y el abono, mirado solo, se queda con parte del crédito del riego (**+2.73 t/ha**, la verdad era +3.00). Mensaje: **el aporte de un ingrediente depende de qué otros ingredientes pongas en la receta**.

---
## 2. Dos Cocineros que Dicen Lo Mismo 👩‍🍳👨‍🍳

La **altitud** y la **temperatura** de la finca van casi pegadas: más arriba, más frío. Si metes las dos en la receta, el modelo **no sabe a cuál darle el crédito**. Para verlo, repetimos el análisis **300 veces** con fincas escogidas al azar (con reemplazo) y miramos **cuánto bailan** los aportes de la altitud con y sin la temperatura.

In [ ]:
print(f"Parecido entre altitud y temperatura (correlacion): {np.corrcoef(fincas.altitud, fincas.temperatura)[0, 1]:.3f}")
b = np.random.default_rng(1)
aporte_sin, aporte_con = [], []
for _ in range(300):
    muestra = fincas.iloc[b.integers(0, n, n)]
    aporte_sin.append(ajusta(["abono", "lluvia", "riego", "altitud"], muestra)["altitud"])
    aporte_con.append(ajusta(["abono", "lluvia", "riego", "altitud", "temperatura"], muestra)["altitud"])
aporte_sin, aporte_con = np.array(aporte_sin), np.array(aporte_con)
print(f"Aporte de la altitud SIN la temperatura: media {aporte_sin.mean():+.4f}, baila (desv. est.) {aporte_sin.std():.4f}   (verdadero: -0.0040)")
print(f"Aporte de la altitud CON la temperatura: media {aporte_con.mean():+.4f}, baila (desv. est.) {aporte_con.std():.4f}   <- {aporte_con.std() / aporte_sin.std():.1f} veces mas inestable")

plt.figure(figsize=(9, 3.8))
plt.hist(aporte_sin, bins=30, alpha=0.7, color="#16a34a", label="sin temperatura"); plt.hist(aporte_con, bins=30, alpha=0.7, color="#dc2626", label="con temperatura")
plt.axvline(-0.004, color="black", ls="--"); plt.xlabel("aporte de la altitud al cambiar de muestra"); plt.ylabel("veces"); plt.title("Dos ingredientes casi iguales: el aporte baila"); plt.legend(); plt.tight_layout(); plt.show()

assert aporte_con.std() > 4 * aporte_sin.std()

---
**Lección:** la altitud y la temperatura se parecen casi por completo (correlación **−0.993**). Sin la temperatura, el aporte de la altitud es estable: **−0.0036** (la verdad era −0.0040) y baila apenas 0.0007 de una muestra a otra. Con la temperatura en la receta, el aporte promedio de la altitud sale **+0.0124** (¡con **signo contrario** al verdadero!) y baila **8.7 veces más** (0.0065). El modelo seguiría prediciendo bien, pero ya **no puede decir cuánto aporta cada una**. Solución: dejar solo una (la más fácil de interpretar) o combinarlas.

---
## 3. Un Solo Dato Raro Mueve la Línea 🔧

Alguien digitó **420 kg** de abono en una finca donde fueron **42**. Un solo número mal puesto, lejos de los demás, puede **arrastrar la línea** hacia él. Comparamos la línea con y sin esa fila.

In [ ]:
sucias = fincas.copy()
i_raro = 0
sucias.loc[i_raro, "abono"] = 420.0                                         # error de digitacion: eran 42

pend_limpia = ajusta(["abono"], fincas)["abono"]
pend_sucia = ajusta(["abono"], sucias)["abono"]
print(f"Aporte del abono (cada 10 kg) con datos limpios: +{10 * pend_limpia:.3f} t/ha | con UN dato mal digitado: +{10 * pend_sucia:.3f} t/ha")

plt.figure(figsize=(9, 3.8))
plt.scatter(sucias.abono, sucias.cosecha, s=18, color="#94a3b8"); plt.scatter(sucias.abono[i_raro], sucias.cosecha[i_raro], color="#dc2626", s=90, marker="X", label="dato mal digitado")
xs = np.array([20, 430]); b0 = ajusta(["abono"], sucias)["base"]; plt.plot(xs, b0 + pend_sucia * xs, color="#dc2626", ls="--", label="linea con el dato malo")
b0l = ajusta(["abono"], fincas)["base"]; plt.plot(xs, b0l + pend_limpia * xs, color="black", label="linea limpia")
plt.xlabel("abono (kg/ha)"); plt.ylabel("cosecha (t/ha)"); plt.legend(); plt.title("Un punto lejano tira de la linea"); plt.tight_layout(); plt.show()

assert pend_sucia < 0.6 * pend_limpia

---
##### 🛠️ Práctica 1: Encontrar el dato raro y comparar

Con el DataFrame `sucias` (150 fincas, una con el abono mal digitado) y la función `ajusta` de las secciones anteriores:

1. Encuentra las filas con un abono **mayor a 3 veces la mediana**: `raras = sucias.index[sucias["abono"] > 3 * sucias["abono"].median()]`.
2. Quita esas filas (`sin_raras = sucias.drop(raras)`) y calcula el aporte del abono en `pend_sin_raras = ajusta(["abono"], sin_raras)["abono"]`.
3. Comprueba con `assert` que se detecta **exactamente una** fila y que `pend_sin_raras` queda dentro de 0.01 de `pend_limpia` (la línea de los datos que no estaban dañados, calculada arriba).
4. Comenta: ¿es buena idea borrar el dato sin averiguar nada? ¿Qué harías primero?

In [ ]:
# Escribe tu código aquí

# raras = ...
# sin_raras = ...
# pend_sin_raras = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
raras = sucias.index[sucias["abono"] > 3 * sucias["abono"].median()]
sin_raras = sucias.drop(raras)
pend_sin_raras = ajusta(["abono"], sin_raras)["abono"]
print(f"Filas raras: {list(raras)} | aporte sin ellas: +{10 * pend_sin_raras:.3f} t/ha por cada 10 kg (con los datos limpios era +{10 * pend_limpia:.3f})")

assert len(raras) == 1 and abs(pend_sin_raras - pend_limpia) < 0.01
print("Antes de borrar, hay que preguntar: ¿fue un error al anotar (se corrige con la fuente) o una finca realmente extrema? Se anota lo que se decidio.")
```
</details>

---

---
**Lección:** con los datos limpios, el abono aporta **+0.81 t/ha por cada 10 kg**; con **un solo número mal digitado** (420 en vez de 42) el aporte baja casi a la mitad: **+0.43 t/ha**. Una finca lejana de las demás tiene mucha «palanca» sobre la línea. Al quitarla (Práctica 1) el aporte vuelve a **+0.81 t/ha**; pero primero se **investiga** el dato, y se anota lo que se decidió.

---
## Resumen en una frase 🎯

> **En una receta con varios ingredientes, el aporte de cada uno se mide dejando los demás quietos (y cambia si olvidas uno que va de la mano con el otro), dos ingredientes casi iguales hacen que los aportes bailen, y un solo dato mal anotado puede mover la línea: hay que revisarlo antes de creerle a los números.**

### Lo que aprendiste hoy:
- ✅ **Aporte de cada ingrediente** «dejando los demás igual»; mirar solo el abono **exagera** su aporte porque se lleva el crédito del riego.
- ✅ **Ingredientes repetidos** (altitud y temperatura): predicen bien, pero **no se puede repartir el crédito** y los aportes se vuelven inestables.
- ✅ **Datos raros:** uno solo puede mover la línea; se **investiga** antes de borrarlo.

### 🧠 ¿Ya lo tienes claro?

1. Un estudio dice «tomar café se asocia con vivir más», pero los que toman café también hacen más ejercicio y tienen más dinero. ¿Cómo usarías la idea de la «receta» para dudar de la conclusión?
2. Si te dicen que dos ingredientes de tu modelo son casi la misma cosa, ¿qué harías: dejar los dos, dejar uno solo o mezclarlos? ¿De qué depende?
3. Encuentras una finca con un abono diez veces más alto que las demás. Menciona tres cosas que harías antes de decidir si la quitas.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>